### Notebook
- **Name:** `cs_get_dem.ipynb`
- **Created/updated:** 2026-02-27
- **Python:** 3.x

### Purpose
Download or prepare the DEM (digital elevation model) required by subsequent steps.

### Inputs
Line trace (`.shp`) and configuration parameters (extent expansion, DEM resolution/source).

### Outputs
Downloaded DEM files and metadata saved under the project data directory.

### Dependencies
- (see imports below)

### Usage
Executed by the project pipeline (e.g., via Papermill) or run interactively in Jupyter.

### Notes
- Keep paths and parameters centralized in `config.toml` / `CONFIG_PATH` where applicable.


In [7]:
import sys
print("Python version")
print(sys.version)
print("Ejecutable")
print(sys.executable)


Python version
3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Ejecutable
c:\Python\Python312\python.exe


In [8]:
from pyproj import Transformer
import subprocess

In [9]:
def utm_rect_to_fetch_dem_bbox(corners_xy, epsg_utm):
    """
    corners_xy: iterable con 4 tuplas (x, y) UTM [m]
    epsg_utm: EPSG del sistema UTM de entrada (p.ej. 25830, 32630)
    Devuelve: (north, east, south, west) en grados (lon/lat WGS84)
    """
    t = Transformer.from_crs(f"EPSG:{epsg_utm}", "EPSG:4326", always_xy=True)

    lons, lats = [], []
    for x, y in corners_xy:
        lon, lat = t.transform(x, y)
        lons.append(lon)
        lats.append(lat)

    north = max(lats)
    south = min(lats)
    east  = max(lons)
    west  = min(lons)
    return north, east, south, west

In [10]:
# --- Ejemplo de uso ---
epsg_utm = 25830
corners_xy = [
    (x1, y1),
    (x2, y2),
    (x3, y3),
    (x4, y4),
]

north, east, south, west = utm_rect_to_fetch_dem_bbox(corners_xy, epsg_utm)

NameError: name 'x1' is not defined

In [ ]:


# --- Entradas ---
#epsg_utm = 25830          # cambia según tu caso (p.ej. 32630, 25829, etc.)
#x, y = 450000, 4800000    # tus coordenadas UTM (m)
#buf_x_km = 5              # semiancho (km)
#buf_y_km = 5              # semialto  (km)

# UTM -> lon/lat (WGS84 geográficas)
#t = Transformer.from_crs(f"EPSG:{epsg_utm}", "EPSG:4326", always_xy=True)
#lon, lat = t.transform(x, y)

# --- Llamada a fetch_dem ---
# Nota: el orden que exige fetch_dem es (lon, lat, buf_x, buf_y). :contentReference[oaicite:1]{index=1}
cmd = [
    "fetch_dem",
    #"--point", str(lon), str(lat), str(buf_x_km), str(buf_y_km),
    "--bbox", str(north), str(east), str(south), str(west),
    "--buf_units", "kilometers",
    "--src", "srtm",
    "--out_res", "30",
    "--fill_no_data",
    "dem.tif"
]

p = subprocess.run(cmd, capture_output=True, text=True)

print("returncode:", p.returncode)
print("STDOUT:\n", p.stdout)
print("STDERR:\n", p.stderr)

if p.returncode != 0:
    raise RuntimeError("fetch_dem falló; revisa STDERR arriba.")


returncode: 0
STDOUT:
 
STDERR:
 
